# Analyse Météo & Climat : Tableau de Bord Gold

In [2]:
!pip install matplotlib seaborn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 56.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 51.7 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 61.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 62.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 62.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [seaborn]9/10 [seaborn]ib]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [5]:
!pip install pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 58.7 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 48.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [fastparquet] [fastparquet]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import os
os.environ["HADOOP_USER_NAME"] = "root"

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max, min, avg, round, date_format


# Configuration du style des graphiques
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)


# 1. Connexion Spark & Chargement des Tables Gold

In [9]:
# Initialisation de la SparkSession connectée au NameNode HDFS
spark = (
    SparkSession.builder
    .appName("MeteoNotebookAnalytics")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

# Lecture des tables Parquet Gold directement depuis HDFS
df_daily = spark.read.parquet("hdfs://namenode:9000/gold/meteo_daily_agg")
df_alerts = spark.read.parquet("hdfs://namenode:9000/gold/meteo_alerts")

# Conversion en DataFrames Pandas pour la visualisation
pdf_daily = df_daily.orderBy("date").toPandas()
pdf_alerts = df_alerts.toPandas()

print(f"[OK] Chargement réussi depuis HDFS ! Jours agrégés : {len(pdf_daily)}")

Py4JJavaError: An error occurred while calling None.org.apache.spark.sql.classic.SparkSession.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.DirectConstructorHandleAccessor.newInstance(DirectConstructorHandleAccessor.java:62)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:499)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:483)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184)
py4j.ClientServerConnection.run(Client


# 2. Cartes KPI (Aperçu Global)



In [ ]:
# Conversion en Pandas pour la dataviz
pdf_daily = df_daily.orderBy("date").toPandas()

if not pdf_daily.empty:
    latest_row = pdf_daily.iloc[-1]
    
    print("=" * 40)
    print(f"  KPI MÉTÉO DU RÉCENT : {latest_row['date']}")
    print("=" * 40)
    print(f"• Température Moyenne : {latest_row['avg_temperature']} °C")
    print(f"• Température Min / Max: {latest_row['min_temperature']} °C / {latest_row['max_temperature']} °C")
    print(f"• Humidité Moyenne    : {latest_row['avg_humidity']} %")
    print(f"• Anomalie thermique   : {latest_row['temp_anomaly']} °C (par rapport à la normale 2023)")
    print("=" * 40)


# 3. Graphique : Évolution Température vs Normale Historique

In [ ]:

plt.figure(figsize=(14, 6))

# Courbe des températures quotidiennes
plt.plot(pdf_daily["date"], pdf_daily["avg_temperature"], label="Température Moyenne (°C)", color="#d9534f", linewidth=2, marker="o")

# Courbe de référence historique
plt.plot(pdf_daily["date"], pdf_daily["hist_avg_temp_month"], label="Moyenne Mensuelle Historique 2023 (°C)", color="#0275d8", linestyle="--", linewidth=2)

plt.title("Évolution de la Température Réelle vs Normale Historique (2023)", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Température (°C)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

# 4. Graphique : Anomalies de Température (°C)

In [ ]:
plt.figure(figsize=(14, 5))

colors = ["#d9534f" if x >= 0 else "#0275d8" for x in pdf_daily["temp_anomaly"]]
plt.bar(pdf_daily["date"].astype(str), pdf_daily["temp_anomaly"], color=colors)

plt.axhline(0, color="black", linestyle="-", linewidth=0.8)
plt.title("Anomalies Thermiques Quotidiennes (Écart vs Historique)", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Écart (°C)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 5. Bilan des Alertes Météo

In [ ]:
pdf_alerts = df_alerts.toPandas()

if not pdf_alerts.empty:
    print("\n--- DÉTAIL DES ALERTES DÉTECTÉES ---")
    print(pdf_alerts[["station_id", "timestamp", "temperature", "precipitation", "alert_gel", "alert_canicule", "alert_pluie_forte"]])
else:
    print("\n[OK] Aucune alerte météo extrême enregistrée.")